# RLS GRANT — обвязка DRP / Impala

SQL Lab режет `current_user` и не даёт SELECT на `rls_acq_user`.
Отсюда: кто ты в GP, GRANT, проверка чтения ACL.

Impala для GRANT не нужен — ячейка опциональная (тот же keytab, что в `final_script_2_1`).


In [ ]:
import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)

NOTEBOOK_REV = '2026-09-25-rls-grant'
drp_schema = 'sbx_da'
drp_superset_grant_role = 'raisa_superset'

ACL_TABLES = [
    'rls_acq_user',
    'rls_acq_user_filial',
    'rls_acq_role_sheet',
]

# Кого наделить SELECT. Смешанный регистр / дефис — в кавычках внутри GRANT.
grant_roles = [
    drp_superset_grant_role,
    'Shestopalov-VYur',
]

run_impala = False  # True — probe Impala тем же keytab

print('rev', NOTEBOOK_REV)
print('grant_roles', grant_roles)
print('tables', [f'{drp_schema}.{t}' for t in ACL_TABLES])


## 1) DRP


In [ ]:
if 'drp' in globals() and drp is not None:
    print('Reuse DRP')
else:
    drp_user = input('DRP user: ').strip()
    drp_password = getpass.getpass('DRP password: ')
    drp = connect(
        to='DRP',
        user_params={'user_name': drp_user, 'password': drp_password},
    )
    print('DRP connected as', drp_user)


def pg_ident(name):
    return '"' + str(name).replace('"', '""') + '"'


who = drp.fetch(
    'SELECT current_user AS current_user, session_user AS session_user'
)
print('=== кто ты в Greenplum (здесь можно, в SQL Lab — нет) ===')
display(who)


## 2) GRANT SELECT на ACL

Владелец таблиц = тот, кто гонял `rls_acq_roles_build`. Если GRANT упадёт — зайди тем же логином.


In [ ]:
with drp:
    try:
        drp.execute('GRANT USAGE ON SCHEMA ' + drp_schema + ' TO ' + pg_ident(drp_superset_grant_role))
        print('OK GRANT USAGE ON SCHEMA', drp_schema, 'TO', drp_superset_grant_role)
    except Exception as exc:
        print('FAIL USAGE', type(exc).__name__, str(exc)[:300])

    for t in ACL_TABLES:
        fq = drp_schema + '.' + t
        for role in grant_roles:
            sql = 'GRANT SELECT ON TABLE ' + fq + ' TO ' + pg_ident(role)
            try:
                drp.execute(sql)
                print('OK', sql)
            except Exception as exc:
                print('FAIL', sql, type(exc).__name__, str(exc)[:300])


## 3) Проверка чтения


In [ ]:
with drp:
    for t in ACL_TABLES:
        fq = drp_schema + '.' + t
        n = drp.fetch('SELECT COUNT(*) AS n FROM ' + fq)
        print(fq, 'rows=', int(pd.to_numeric(n.iloc[0, 0], errors='coerce')))
    display(drp.fetch(
        'SELECT username, role, is_all_filials FROM '
        + drp_schema + '.rls_acq_user ORDER BY 1'
    ))

print('Дальше в SQL Lab — только SELECT из vd_acq_rls_overview, без current_user.')


## 4) Impala (по желанию)


In [ ]:
if not run_impala:
    print('SKIP Impala (run_impala=False)')
else:
    if 'imp' in globals() and imp is not None:
        print('Reuse Impala')
    else:
        imp = connect(
            to='IMPALA',
            extra_options={'db': 'sandbox_ai'},
            driver_args={'tez.queue.name': 'ai'},
            kerberos={
                'keytab_path': '/home/jovyan/test_requests/tech.keytab',
                'use_credentials': True,
                'update_keytab': True,
            },
            user_params={'user_name': 'Shestopalov-VYur'},
        )
        imp._init_connection()
        print('Impala connected')
    with imp:
        try:
            imp.execute('set MEM_LIMIT=4g')
        except Exception:
            pass
        probe = imp.fetch('SELECT 1 AS ok')
    display(probe)
